# 2 · Experten: Rolling-Origin-Prognosen (Stufe 1)

Jeder Experte prognostiziert für jeden Ursprung $t$ (letzter bekannter Tag) und Horizont $h$ die RV am Tag $t+h$, als Mittelwert und 21 Quantile. Er sieht nur Daten bis einschliesslich $t$.

Die Prognosen sind teuer (Foundation Models) und werden pro Serie und Modell als `<SID>__<Modell>.parquet` gecacht. **Vorhandene Dateien werden nur geladen, nie überschrieben.** Für `binance` liegen Chronos-2 und TimesFM-2.5 schon in `results/`.

| Experte | Input | Idee |
|---|---|---|
| **RW** | log-RV | „morgen wie heute“, Streuung aus historischen h-Tages-Änderungen |
| **EWMA** | log-RV | exponentiell geglättete log-RV (Halbwertszeit 5 Tage) |
| **HAR** | log-RV | $\ln RV_{t+h} = \beta_0 + \beta_1 \ln RV_t + \beta_2 \ln RV^{(5)}_t + \beta_3 \ln RV^{(22)}_t$, OLS im 1000-Tage-Fenster |
| **GARCH** | Renditen | GARCH(1,1)-t, über Log-Residuen auf die RV kalibriert |
| **Chronos-2-full** | log-RV | Foundation Model (Amazon), zero-shot, Kontext 1024 |
| **TimesFM-2.5** | log-RV | Foundation Model (Google), zero-shot, 9 Quantile → 21 ergänzt |

Ökonometrische Modelle: Quantile aus den empirischen Log-Residuen, Mittelwert per Smearing (Duan 1983). Foundation Models: Quantile direkt, Mittelwert durch Integration der Quantile.

**Neuen Experten hinzufügen:** Klasse mit `name` und `forecast(s, origins)` schreiben, in `EXPERTS` eintragen, Namen in `MODELS`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASET = "binance"        # Name aus 01_daten.ipynb, z.B. "binance" oder "syn_regime_signal_s0"

DATA_DIR, RESULTS_DIR = Path("data"), Path("results")
INFO = json.loads((DATA_DIR / f"{DATASET}.json").read_text())
DAILY_DIR, FORECAST_DIR = Path(INFO["daily_dir"]), Path(INFO["forecast_dir"])
SIDS = INFO["sids"]
SYNTHETIC = INFO["info"]["source"] == "synthetic"
print(f"Datensatz {DATASET}: {len(SIDS)} Serien {SIDS}")

import time
from statistics import NormalDist
from tqdm.auto import tqdm

QUANTILES = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5,
             0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]   # = native Chronos-2-Quantile
QCOLS = [f"q{q:.2f}" for q in QUANTILES]
_LV = np.asarray(QUANTILES)

_QMASS = (np.r_[_LV[1:], 1.0] - np.r_[0.0, _LV[:-1]]) / 2     # Wahrscheinlichkeitsmasse je Quantil (Trapez)
_QMASS = _QMASS / _QMASS.sum()

MODELS = ["RW", "EWMA", "HAR", "GARCH", "Chronos-2-full", "TimesFM-2.5"]
SKIP_MISSING_PACKAGES = True    # Foundation Model nicht installiert -> überspringen statt abbrechen

HORIZONS = [1, 5]               # Tage voraus
MIN_HISTORY = 500               # Tage Historie vor dem ersten Ursprung
EST_WINDOW = 1000               # rollierendes Schätzfenster für RW / EWMA / HAR / GARCH
GARCH_REFIT_EVERY = 5           # GARCH alle k Ursprünge neu schätzen
CONTEXT = 1024                  # Kontext der Foundation Models (Tage)
FM_BATCH = 128

Datensatz binance: 3 Serien ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']


## Serien und Ursprünge

In [2]:
class RVSeries:
    def __init__(self, sid, df):
        self.sid = sid
        self.dates = df.index
        self.rv = df["rv"].to_numpy(float)
        self.ret = df["ret"].to_numpy(float)
        self.logrv = np.log(self.rv)
        self.origins = np.arange(MIN_HISTORY - 1, len(df) - max(HORIZONS))


series = {sid: RVSeries(sid, pd.read_parquet(DAILY_DIR / f"{sid}_daily.parquet")) for sid in SIDS}
for s in series.values():
    print(f"{s.sid}: {len(s.origins)} Ursprünge ab {s.dates[s.origins[0]].date()}")

BTCUSDT: 1922 Ursprünge ab 2021-05-21
ETHUSDT: 1922 Ursprünge ab 2021-05-21
SOLUSDT: 1702 Ursprünge ab 2021-12-29


## Ökonometrische Experten

In [3]:
def dist_from_log_resid(log_point, resid):
    # Log-Punktprognose + empirische Log-Residuen -> (Mittelwert, Quantile) auf RV-Skala
    resid = resid[np.isfinite(resid)]
    q = np.exp(log_point + np.quantile(resid, _LV))
    mean = np.exp(log_point) * np.mean(np.exp(resid))            # Smearing-Schätzer
    return mean, q


def mean_from_quantiles(q):
    return q @ _QMASS                                            # E[Y] als Integral der Quantilfunktion


class Expert:
    name = "base"

    def forecast(self, s, origins):
        raise NotImplementedError

    @staticmethod
    def _alloc(n):
        return (np.full((n, len(HORIZONS)), np.nan),
                np.full((n, len(HORIZONS), len(QUANTILES)), np.nan))


class RandomWalk(Expert):
    name = "RW"

    def forecast(self, s, origins):
        L = s.logrv
        mean, q = self._alloc(len(origins))
        for i, t in enumerate(origins):
            lo = max(0, t - EST_WINDOW + 1)
            for j, h in enumerate(HORIZONS):
                resid = L[lo + h: t + 1] - L[lo: t + 1 - h]
                mean[i, j], q[i, j] = dist_from_log_resid(L[t], resid)
        return mean, q


class EWMA(Expert):
    name = "EWMA"

    def __init__(self, halflife=5):
        self.halflife = halflife

    def forecast(self, s, origins):
        L = s.logrv
        sm = pd.Series(L).ewm(halflife=self.halflife).mean().to_numpy()   # kausal: nur Daten bis t
        mean, q = self._alloc(len(origins))
        for i, t in enumerate(origins):
            lo = max(0, t - EST_WINDOW + 1)
            for j, h in enumerate(HORIZONS):
                resid = L[lo + h: t + 1] - sm[lo: t + 1 - h]
                mean[i, j], q[i, j] = dist_from_log_resid(sm[t], resid)
        return mean, q


class HAR(Expert):
    # log-HAR (Corsi 2009): direkte Prognose je Horizont, OLS im rollierenden Fenster
    name = "HAR"

    def forecast(self, s, origins):
        rv = pd.Series(s.rv)
        X = np.column_stack([np.ones(len(rv)), np.log(rv),
                             np.log(rv.rolling(5).mean()), np.log(rv.rolling(22).mean())])
        L = s.logrv
        mean, q = self._alloc(len(origins))
        for i, t in enumerate(origins):
            lo = max(21, t - EST_WINDOW + 1)
            for j, h in enumerate(HORIZONS):
                rows = np.arange(lo, t - h + 1)        # nur Zeilen, deren Ziel u+h <= t schon bekannt ist
                beta, *_ = np.linalg.lstsq(X[rows], L[rows + h], rcond=None)
                resid = L[rows + h] - X[rows] @ beta
                mean[i, j], q[i, j] = dist_from_log_resid(X[t] @ beta, resid)
        return mean, q


class GARCH(Expert):
    # GARCH(1,1)-t auf Tagesrenditen. Niveau und Streuung über log(RV_u / sigma²_u) auf die RV kalibriert.
    name = "GARCH"

    def __init__(self, dist="t", refit_every=GARCH_REFIT_EVERY):
        self.dist, self.refit_every = dist, refit_every

    def forecast(self, s, origins):
        import warnings
        from arch import arch_model
        warnings.filterwarnings("ignore", module="arch")

        y_all = 100 * s.ret                       # Prozent-Renditen für eine stabile Optimierung
        mean, q = self._alloc(len(origins))
        params = None
        for i, t in enumerate(origins):
            lo = max(0, t - EST_WINDOW + 1)
            am = arch_model(y_all[lo: t + 1], mean="Constant", vol="GARCH", p=1, q=1,
                            dist=self.dist, rescale=False)
            res = None
            if params is None or i % self.refit_every == 0:
                try:
                    res = am.fit(disp="off", show_warning=False,
                                 starting_values=None if params is None else params.to_numpy())
                    params = res.params
                except Exception:
                    res = None
            if res is None:                        # zwischen Refits: alte Parameter auf neue Daten
                res = am.fix(params)
            sig2_in = np.asarray(res.conditional_volatility) ** 2 / 1e4
            sig2_fc = res.forecast(horizon=max(HORIZONS), reindex=False).variance.to_numpy()[-1] / 1e4
            resid = s.logrv[lo: t + 1] - np.log(sig2_in)
            for j, h in enumerate(HORIZONS):
                mean[i, j], q[i, j] = dist_from_log_resid(np.log(sig2_fc[h - 1]), resid)
        return mean, q

## Foundation Models

Beide laufen zero-shot auf der log-RV mit 1024 Tagen Kontext. Sie werden erst importiert, wenn sie wirklich rechnen müssen.

- **Chronos-2:** `cross_learning=False`, sonst teilen Kontexte im Batch Information (Leakage).
- **TimesFM-2.5** liefert nur die Quantile 0.1 … 0.9. Die übrigen werden **in Logs** auf der Normal-Skala ergänzt: innen linear, an den Rändern mit der lokalen Steigung. Das ist exakt, falls die log-RV normalverteilt ist (Kontrolle unten).

In [4]:
DEVICE = None    # None = cuda falls verfügbar, sonst cpu


def _device():
    import torch
    return DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")


def contexts_for(s, origins):
    L = s.logrv.astype(np.float32)
    return [L[max(0, t - CONTEXT + 1): t + 1] for t in origins]


class Chronos2(Expert):
    def __init__(self, model_id="amazon/chronos-2", name="Chronos-2-full", chunk=1024):
        from chronos import Chronos2Pipeline
        self.name, self.chunk = name, chunk
        self.pipe = Chronos2Pipeline.from_pretrained(model_id, device_map=_device())

    def forecast(self, s, origins):
        ctx = contexts_for(s, origins)
        preds = []
        for k in tqdm(range(0, len(ctx), self.chunk), desc=f"{self.name} {s.sid}", leave=False):
            qs, _ = self.pipe.predict_quantiles(
                ctx[k: k + self.chunk], prediction_length=max(HORIZONS), quantile_levels=QUANTILES,
                batch_size=FM_BATCH, context_length=CONTEXT,
                cross_learning=False)       # sonst teilen Kontexte im Batch Information -> Leakage
            preds += [p[0].detach().float().cpu().numpy() for p in qs]
        log_q = np.stack(preds)[:, [h - 1 for h in HORIZONS], :]
        q = np.sort(np.exp(log_q), axis=-1)
        return mean_from_quantiles(q), q


_ND = NormalDist()
NATIVE9 = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
_Z9 = np.array([_ND.inv_cdf(p) for p in NATIVE9])
_Z21 = np.array([_ND.inv_cdf(p) for p in QUANTILES])


def expand_quantiles(logq9):
    # (..., 9) Log-Quantile auf 0.1..0.9 -> (..., 21) Log-Quantile auf QUANTILES
    q = np.asarray(logq9, float)
    out = np.empty(q.shape[:-1] + (len(QUANTILES),))
    lo_slope = np.maximum((q[..., 2] - q[..., 0]) / (_Z9[2] - _Z9[0]), 1e-6)
    hi_slope = np.maximum((q[..., 8] - q[..., 6]) / (_Z9[8] - _Z9[6]), 1e-6)
    for k, z in enumerate(_Z21):
        if z < _Z9[0] - 1e-12:
            out[..., k] = q[..., 0] + lo_slope * (z - _Z9[0])
        elif z > _Z9[-1] + 1e-12:
            out[..., k] = q[..., 8] + hi_slope * (z - _Z9[-1])
        else:
            j = int(np.clip(np.searchsorted(_Z9, z), 1, 8))
            w = (z - _Z9[j - 1]) / (_Z9[j] - _Z9[j - 1])
            out[..., k] = (1 - w) * q[..., j - 1] + w * q[..., j]
    return out


nd = NormalDist(-7.0, 0.6)    # Kontrolle: bei normalverteilter log-RV muss die Ergänzung exakt sein
print("max. Fehler der Quantil-Ergänzung bei Normalverteilung:",
      float(np.abs(expand_quantiles(np.array([nd.inv_cdf(p) for p in NATIVE9]))
                   - np.array([nd.inv_cdf(p) for p in QUANTILES])).max()))


class TimesFM25(Expert):
    name = "TimesFM-2.5"

    def __init__(self, repo="google/timesfm-2.5-200m-pytorch", chunk=1024):
        import timesfm
        self.chunk = chunk
        self.model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(repo)
        self.model.compile(timesfm.ForecastConfig(
            max_context=CONTEXT, max_horizon=max(HORIZONS), normalize_inputs=True,
            per_core_batch_size=FM_BATCH,        # Default 1 wäre ca. 8x langsamer
            use_continuous_quantile_head=True, fix_quantile_crossing=True,
            infer_is_positive=False))            # log-RV ist negativ

    def forecast(self, s, origins):
        ctx = contexts_for(s, origins)
        parts = []
        for k in tqdm(range(0, len(ctx), self.chunk), desc=f"TimesFM {s.sid}", leave=False):
            # list(...) nötig: forecast() hängt Padding an die übergebene Liste an
            _, q = self.model.forecast(horizon=max(HORIZONS), inputs=list(ctx[k: k + self.chunk]))
            parts.append(np.asarray(q)[:, :max(HORIZONS), 1:])    # Index 0 = Punktwert, 1..9 = q0.1..q0.9
        logq9 = np.concatenate(parts)[:, [h - 1 for h in HORIZONS], :]
        q = np.sort(np.exp(expand_quantiles(logq9)), axis=-1)
        return mean_from_quantiles(q), q


EXPERTS = {
    "RW": RandomWalk,
    "EWMA": EWMA,
    "HAR": HAR,
    "GARCH": GARCH,
    "Chronos-2-full": lambda: Chronos2("amazon/chronos-2", "Chronos-2-full"),
    "Chronos-2": lambda: Chronos2("autogluon/chronos-2-small", "Chronos-2"),
    "TimesFM-2.5": TimesFM25,
}

max. Fehler der Quantil-Ergänzung bei Normalverteilung: 8.881784197001252e-16


## Prognosen erzeugen (mit Cache)

Eine Zeile pro Ursprung und Horizont mit Ziel `y` (gemessene RV), `mean` und den 21 Quantilen. Laufzeiten landen in `runtimes.csv` im Prognose-Ordner.

In [5]:
def forecast_path(sid, model):
    return FORECAST_DIR / f"{sid}__{model}.parquet"


def run_expert(expert, s):
    t0 = time.time()
    mean, q = expert.forecast(s, s.origins)
    runtime = time.time() - t0
    parts = []
    for j, h in enumerate(HORIZONS):
        tgt = s.origins + h
        df = pd.DataFrame({"sid": s.sid, "model": expert.name, "h": h,
                           "origin": s.dates[s.origins], "target_date": s.dates[tgt],
                           "y": s.rv[tgt], "mean": mean[:, j]})
        parts.append(pd.concat([df, pd.DataFrame(q[:, j, :], columns=QCOLS)], axis=1))
    pd.concat(parts, ignore_index=True).to_parquet(forecast_path(s.sid, expert.name))
    with open(FORECAST_DIR / "runtimes.csv", "a") as f:
        f.write(f"{s.sid},{expert.name},{runtime:.1f}\n")
    print(f"{expert.name:>16} | {s.sid}: {runtime:.0f}s")


FORECAST_DIR.mkdir(parents=True, exist_ok=True)
for m in MODELS:
    todo = [sid for sid in SIDS if not forecast_path(sid, m).exists()]
    if not todo:
        print(f"{m:>16} | gecacht")
        continue
    try:
        expert = EXPERTS[m]()
    except ImportError as e:
        if not SKIP_MISSING_PACKAGES:
            raise
        print(f"{m:>16} | übersprungen, Paket fehlt ({e.name})")
        continue
    for sid in todo:
        run_expert(expert, series[sid])
    del expert

              RW | gecacht
            EWMA | gecacht
             HAR | gecacht
           GARCH | gecacht
  Chronos-2-full | gecacht
     TimesFM-2.5 | gecacht


## Überblick: welche Prognosen gibt es?

In [6]:
rows = []
for p in sorted(FORECAST_DIR.glob("*__*.parquet")):
    sid, m = p.stem.split("__")
    if sid in SIDS:
        f = pd.read_parquet(p, columns=["h", "origin"])
        rows.append({"Serie": sid, "Modell": m, "Prognosen (h=1)": int((f["h"] == 1).sum()),
                     "von": f["origin"].min().date(), "bis": f["origin"].max().date()})
display(pd.DataFrame(rows).pivot_table(index="Modell", columns="Serie", values="Prognosen (h=1)"))

Serie,BTCUSDT,ETHUSDT,SOLUSDT
Modell,,,
Chronos-2,1922.0,1922.0,1702.0
Chronos-2-full,1922.0,1922.0,1702.0
EWMA,1922.0,1922.0,1702.0
GARCH,1922.0,1922.0,1702.0
HAR,1922.0,1922.0,1702.0
RW,1922.0,1922.0,1702.0
TimesFM-2.5,1922.0,1922.0,1702.0
